**Previously on:** Lecture 28 asked whether a difference between two groups was real or
noise -- the last of five separate modeling questions Module 2 has taught since Lecture 19.
Today we chain all five into one pipeline, on one dataset, start to finish -- your map for
what Midterm 2 will ask you to do by hand, on paper, with a small dataset in front of you.

Today's topics:
* the five ideas of Module 2, reframed as one five-step pipeline
* a cautionary cameo: messy data still needs the outlier check first
* one integrated worked example: load -> outlier check -> fit -> evaluate -> hypothesis test
* Quiz 6, in class -- scoped to exactly this pipeline
* paper-exam format rehearsal for Midterm 2

> By the end of today, handed a fresh materials dataset, you should be able to run the full
> pipeline yourself: spot outliers, fit a model, evaluate it honestly, and ask whether a
> difference is real. Quiz 6 is given in class today; HW9 is due; you'll leave with a
> practice Midterm 2 packet for Lecture 30.

# Section 1 -- five questions, one pipeline

Five lectures, five questions -- not five separate topics you pick from, but one pipeline
you walk through, in order, every time you're handed a new dataset:

1. **Lecture 19** -- "What's the simplest line through my data?" (linear regression)
2. **Lecture 20** -- "Do I actually trust that line, or did it just memorize my data?"
   (train/test split, overfitting)
3. **Lectures 22-24** -- "What if the relationship isn't a line, or depends on more than
   one variable?" (`curve_fit`, multivariate regression)
4. **Lecture 27** -- "Is that one weird point ruining everything?" (outliers)
5. **Lecture 28** -- "Is this difference between two groups real, or just noise?"
   (hypothesis testing)

Write the pipeline down -- it is a valid thing to write at the top of your Midterm 2 scratch
paper:

```
look at data -> check for outliers -> fit a model -> evaluate honestly -> test a hypothesis
```

Today we walk it once, live, start to finish, on `steels.csv`. Then you do a compressed
version yourselves on Quiz 6.

# Section 2 -- cameo: messy data still needs the outlier check first

Before today's clean main example, a reminder that real materials data is almost never
handed to you this clean. This is the messy elemental-property table you'll fully clean in
Lectures 32-33 -- don't clean it today, just look at one column.

## Dataset: Metal Alloy Elemental Properties (uncleaned)

In [1]:
import os
import pandas as pd

_file = 'alloy_cleaning.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github, encoding='latin1')
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e
data.head()

,number,name,symbol,name_symbol,pronunciation,appearance,atomic_number,group_block,period,element_category,...,band_gap,recognised_as_an_element_by,curie_point,recognized_as_a_unique_metal_by,recognized_as_a_distinct_element_by,thermal_diffusivity,tensile_strength,molar_volume,proposed_formal_name,alternative_names
0,1,Hydrogen,H,"hydrogen, H","/?ha?dr?d??n/, HY-dr?-j?n",colorless gas,1,"group 1, s-block",1,diatomic nonmetal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Helium,He,"helium, He","/?hi?li?m/, HEE-lee-?m","colorless gas, exhibiting a red-orange glow wh...",2,"group 18 (noble gases), s-block",1,noble gas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Lithium,Li,"lithium, Li","/?l??i?m/, LI-thee-?m",silvery-white,3,"group 1 (alkali metals), s-block",2,alkali metal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Beryllium,Be,"beryllium, Be","/b??r?li?m/, b?-RIL-ee-?m",white-gray metallic,4,"group 2 (alkaline earth metals), s-block",2,alkaline earth metal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Boron,B,"boron, B",/?b??r?n/,black-brown,5,"group 13, p-block",2,metalloid,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
data['tensile_strength'].isna().sum(), len(data)

(np.int64(115), 117)

115 of 117 rows are blank in this column outright -- and the two that *aren't* blank are
still a mess:

In [3]:
data['tensile_strength'].value_counts(dropna=False)

tensile_strength
NaN            115
125240 MPa      1
120              1
Name: count, dtype: int64

One value is a clean number as a string (`'120'`), the other is a corrupted range --
`'125\x96240 MPa'`, where `\x96` is a mis-decoded en dash and `MPa` is a stray unit, both
baked into the same string. One-line demo: try converting the whole column to numbers.

In [4]:
try:
    data['tensile_strength'].astype(float)
except ValueError as e:
    print(f'{type(e).__name__}: {e}')

ValueError: could not convert string to float: '125\x96240 MPa'


Fails immediately, and not gently -- one bad string blocks the entire column, even the 115
rows that are just missing. This is the bridge forward to Lecture 32 ("this is exactly what
we'll fix properly in two weeks") and the bridge backward to Lecture 27 ("and even after a
column like this is numeric, some of its values will be outliers, not typos -- you have to
decide which is which").

Today's main example uses a dataset that's already numeric and clean (`steels.csv`),
specifically so we can spend our time on modeling judgment, not string-wrangling. Keep the
contrast in mind -- it's the whole reason Section 3 opens with an outlier check before
fitting anything, even on "clean" data.

# Section 3 -- the integrated pipeline, live, on `steels.csv`

Same 915-alloy steels dataset since Lecture 15: composition columns `C` through `Nb + Ta` in
weight percent, mechanical properties including `Tensile Strength (MPa)`, and an
`Alloy family` label. Today's question: **can we predict tensile strength from composition,
and do we believe the answer?**

## Dataset: Steel Alloy Compositions and Mechanical Properties

This dataset contains elemental compositions and mechanical properties of 915 steel alloys.

Composition columns (in wt%): `C`, `Si`, `Mn`, `P`, `S`, `Ni`, `Cr`, `Mo`, `Cu`, `V`, `Al`, `N`, `Ceq`, `Nb + Ta`

Target columns: `0.2% Proof Stress (MPa)`, `Tensile Strength (MPa)`, `Elongation (%)`, `Reduction in Area (%)`

The `Alloy code` column contains labels like `A1`, `B3`, etc. We derive `alloy_family` (first letter only)
to get a small set of categorical labels useful for classification tasks (4 families: C, L, M, V, all >150 samples).

In [5]:
import os
import pandas as pd

_file = 'steels.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

# derive alloy family label (first letter of alloy code)
data['Alloy family'] = [c[0] for c in data['Alloy code']]

# set up features (composition columns) and target
x = data.loc[:, ' C':'Nb + Ta']
y = data['Alloy family']
alloy_family = data['Alloy family']

print(f'{len(data)} samples, {x.shape[1]} composition features')
data.head()

915 samples, 14 composition features


,Alloy code,C,Si,Mn,P,S,Ni,Cr,Mo,Cu,...,Al,N,Ceq,Nb + Ta,Temperature (°C),0.2% Proof Stress (MPa),Tensile Strength (MPa),Elongation (%),Reduction in Area (%),Alloy family
0,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,27,342,490,30,71,M
1,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,100,338,454,27,72,M
2,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,200,337,465,23,69,M
3,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,300,346,495,21,70,M
4,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,400,316,489,26,79,M


## (a) Look at the data

In [6]:
data[[' C', ' Mn', ' Cr', ' Tensile Strength (MPa)']].describe()

,C,Mn,Cr,Tensile Strength (MPa)
count,915.000000,915.000000,915.000000,915.000000
mean,0.174929,0.812962,0.427861,496.248087
std,0.059674,0.342775,0.457568,239.710650
min,0.090000,0.420000,0.000000,162.000000
25%,0.130000,0.500000,0.040000,413.000000
50%,0.160000,0.680000,0.110000,479.000000
75%,0.200000,1.210000,1.000000,575.000000
max,0.340000,1.480000,1.310000,6661.000000


Same descriptive-stats vocabulary as Module 1, applied to a bigger question. Also notice
the `Tensile Strength (MPa)` **max**: 6661 MPa -- next to a 75th percentile of 575 MPa, that
number should already look wrong. Section (b) makes that instinct systematic instead of just
eyeballing the table.

## (b) Check for outliers, systematically

Lecture 27's tool, applied before touching any fit: z-scores on
`Tensile Strength (MPa)`, flagged at `|z| > 3`.

In [7]:
ts = data[' Tensile Strength (MPa)']
z = (ts - ts.mean()) / ts.std()
flagged = data[z.abs() > 3]
print(f'{len(flagged)} alloy(s) flagged by |z| > 3 on Tensile Strength')
flagged[['Alloy code', ' Tensile Strength (MPa)']].assign(z=z[z.abs() > 3].round(2))

1 alloy(s) flagged by |z| > 3 on Tensile Strength


,Alloy code,Tensile Strength (MPa),z
626,VbF,6661,25.72


One hit: alloy `VbF`, row 626, `z ≈ 25.7` -- not a borderline case, a spectacular one. This
is the exact row Lecture 19 eyeball-checked and dropped by hand ("Lecture 27 will teach you
to catch things like this systematically"). Today the systematic tool finds it on its own,
no eyeballing required. Its 6661 MPa is almost certainly a data-entry error -- the next-
highest tensile strength in all 915 alloys is under 900 MPa -- so this is category 2 from
Lecture 27's judgment-call taxonomy: a parsing/entry artifact, not real physics. Drop it and
recheck:

In [8]:
data = data.drop(index=626)
z = (data[' Tensile Strength (MPa)'] - data[' Tensile Strength (MPa)'].mean()) / data[' Tensile Strength (MPa)'].std()
flagged_after = data[z.abs() > 3]
print(f'{len(flagged_after)} alloys flagged by |z| > 3 on the remaining {len(data)}')
print(f'largest |z| remaining: {z.abs().max():.3f}')

0 alloys flagged by |z| > 3 on the remaining 914
largest |z| remaining: 2.704


Zero. Same anticlimactic-on-purpose result as Lecture 27's melting-point check: "nothing
flagged" is a legitimate finding, not a failure of the code. The one point that mattered was
the impossible one, and it's gone -- there's no further argument to have about "genuine high-
strength alloy vs. error" today, because the real physics in this column, once the typo is
gone, never crosses the |z| > 3 fence. That won't always be true of every dataset you check --
today it happens to be, and that's worth saying out loud rather than inventing a fight that
isn't there.

## (c) Fit a model

Multivariate linear regression, same `.fit()`/`.predict()`-wrapped least-squares idea as
Lectures 20 and 23 -- six hand-picked composition columns (not all 14) predicting tensile
strength, small enough that the coefficient table stays readable:

In [9]:
import numpy as np


class LinearModel:
    "Wraps lstsq-fitted coefficients with a .predict() method, same shape as Lectures 20/23."

    def __init__(self, coeffs):
        self.coeffs = coeffs

    def predict(self, x):
        X = np.column_stack([np.ones(len(x)), np.asarray(x)])
        return X.dot(self.coeffs)


def fit_linear(x, y):
    "Least-squares fit with an intercept column, via np.linalg.lstsq."
    X = np.column_stack([np.ones(len(x)), np.asarray(x)])
    coeffs, *_ = np.linalg.lstsq(X, np.asarray(y), rcond=None)
    return LinearModel(coeffs)


features = [' C', ' Mn', ' Si', ' Cr', ' Ni', ' Mo']
x = data[features]
y = data[' Tensile Strength (MPa)']

## (d) Evaluate honestly

Split *before* fitting -- Lecture 20's discipline, re-ordered on screen so it reads
split -> fit -> score, not fit -> split:

## Train/Test Split

In order to evaluate the performance of a model on unseen data, we need
to *hold out* some data. This will be called the **test** data.
The data used to fit will be called the **training** data.

As we use more sophisticated models, it is vitally important that we
evaluate our models in this way to avoid *overfitting*.
Remember that a model with enough degrees of freedom can perfectly fit
any data, but it won't have any predictive power!

```
all data:    [########################################################]
shuffle, then slice into two pieces:
train (75%): [##########################################]
test  (25%):                                              [##########]
```

We shuffle the *indices* once, then slice both `x` and `y` with that same shuffled order --
so a row's features and its target always stay paired.

In [10]:
import numpy as np

# split indices rather than data -- more flexible
rng = np.random.default_rng(0)
shuffled = rng.permutation(data.index)
n_test = int(np.ceil(0.25 * len(shuffled)))
idx_test = shuffled[:n_test]
idx_train = shuffled[n_test:]

xtrain = x.loc[idx_train]
xtest  = x.loc[idx_test]

ytrain = y.loc[idx_train]
ytest  = y.loc[idx_test]

print(f"Train: {xtrain.shape[0]} samples, Test: {xtest.shape[0]} samples")

Train: 685 samples, Test: 229 samples


## Convenience Functions

We'll set up some convenience functions for evaluating and visualizing
model performance so we don't have to repeat the same code for every model.

In [11]:
import matplotlib.pyplot as plt


def calc_rmse(model, X, y):
    "Calculate the RMSE from a fitted model."
    y_pred = model.predict(X)
    residuals = y_pred - y
    return np.sqrt(np.mean(residuals**2))


def evaluate_model(model, xtrain, xtest, ytrain, ytest):
    "Evaluate model performance on train and test set, then print the results."
    for name, xi, yi in [('train', xtrain, ytrain), ('test', xtest, ytest)]:
        y_pred = model.predict(xi)
        residuals = y_pred - yi
        # SSE-based, not Var(residuals)-based: Var() silently re-centers on the residual mean,
        # which is ~0 on training data (by construction, for a least-squares fit) but can be far
        # from 0 on held-out data -- inflating R2 exactly when it matters most (test-set scoring).
        sse = np.sum(residuals**2)
        sst = np.sum((yi - yi.mean())**2)
        r2 = 1 - sse / sst
        rmse = np.sqrt(np.mean(residuals**2))
        print(f'{name:5s}: Rsq = {r2:.3f}, RMSE = {rmse:.3f}')


def plot_model(model, xtrain, xtest, ytrain, ytest):
    "Create a parity plot using a trained model with train/test split."
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(ytrain, model.predict(xtrain), '.', label='Training Data')
    ax.plot(ytest, model.predict(xtest), '.', label='Testing Data')
    # min/max come from ytrain/ytest -- the arguments actually passed in -- never a global `y`.
    # Reaching out to a global here would be the exact scope anti-pattern Lecture 14's
    # bad_zscore example warned against.
    lo = min(np.min(ytrain), np.min(ytest))
    hi = max(np.max(ytrain), np.max(ytest))
    min_max = np.array([lo, hi])
    ax.plot(min_max, min_max, 'k--', label='Parity')
    ax.set_aspect('equal')
    ax.set_xlabel('Observation')
    ax.set_ylabel('Prediction')
    ax.legend()

In [12]:
model = fit_linear(xtrain, ytrain)
evaluate_model(model, xtrain, xtest, ytrain, ytest)

train: Rsq = 0.219, RMSE = 110.290
test : Rsq = 0.266, RMSE = 110.508


Composition alone -- six elements, no processing or temperature information -- explains a
modest slice of the variation (R² around 0.2-0.3), well below Lecture 19's 0.70 for proof
stress predicting tensile strength directly. That's not a bug: proof stress already carries
processing history baked into it, while raw composition doesn't. Train and test land close
together here, with test even a touch *above* train -- not the "train comfortably above test"
pattern Lecture 20 called healthy-and-expected, just a different roll of the same die. Lecture
20 Section 6 warned that one split is one arbitrary draw; Lecture 23's "Evaluating honestly"
section showed the fix in action -- sweep a few seeds and watch the number move:

In [13]:
for seed in range(5):
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(data.index)
    n_test = int(np.ceil(0.25 * len(shuffled)))
    idx_te, idx_tr = shuffled[:n_test], shuffled[n_test:]
    m = fit_linear(x.loc[idx_tr], y.loc[idx_tr])
    tr_pred, te_pred = m.predict(x.loc[idx_tr]), m.predict(x.loc[idx_te])
    tr_r2 = 1 - np.sum((tr_pred - y.loc[idx_tr]) ** 2) / np.sum((y.loc[idx_tr] - y.loc[idx_tr].mean()) ** 2)
    te_r2 = 1 - np.sum((te_pred - y.loc[idx_te]) ** 2) / np.sum((y.loc[idx_te] - y.loc[idx_te].mean()) ** 2)
    print(f'seed {seed}: train R2 = {tr_r2:.3f}   test R2 = {te_r2:.3f}')

seed 0: train R2 = 0.219   test R2 = 0.266
seed 1: train R2 = 0.263   test R2 = 0.122
seed 2: train R2 = 0.213   test R2 = 0.285
seed 3: train R2 = 0.247   test R2 = 0.183
seed 4: train R2 = 0.237   test R2 = 0.210


Five seeds, five different train/test pairs, sometimes train ahead, sometimes test ahead --
exactly Lecture 20's point: one split is one roll of the dice, especially once R² itself is
this modest. Now look at the coefficients from the original split, physically:

In [14]:
for name, c in zip(features, model.coeffs[1:]):
    print(f'{name:10s} {c: .4f}')

 C          304.3655
 Mn         168.3145
 Si         24.6402
 Cr         38.7227
 Ni        -67.8852
 Mo         101.5164


Cr and Mo -- classic solid-solution and carbide-forming strengtheners -- both come out
positive, physically sensible. Ni comes out *negative* -- and before you write "nickel
weakens steel" in a report, check it the way Lecture 23 taught: on its own, Ni's
correlation with tensile strength is *positive* (+0.28, the strongest single feature of
these six). The negative multivariate coefficient is a correlated-features artifact: Ni
and Mn travel together in this data (r = 0.45), so the fit can shuffle credit between
them -- drop Mn and refit, and Ni's coefficient flips positive (to roughly +130). Same
lesson as Lecture
23's locked-together printer settings: coefficients answer "holding the others fixed,"
and when features move together, no experiment in this dataset ever *did* hold the
others fixed. A surprising sign is a prompt to check for collinearity, not a discovery.

## (e) Ask a hypothesis question about it

Back in Lecture 15, `groupby('Alloy family')` first hinted that comparing groups "properly,
with a real yes-or-no answer," was a question for later in the course. Later is now. Split
by `Alloy family` into the two most populous families and run Lecture 28's two-sample test
on `Tensile Strength (MPa)`:

In [15]:
from scipy import stats

c_group = data[data['Alloy family'] == 'C'][' Tensile Strength (MPa)']
m_group = data[data['Alloy family'] == 'M'][' Tensile Strength (MPa)']
print(f'family C: n={len(c_group)}, mean={c_group.mean():.2f} MPa, std={c_group.std():.2f} MPa')
print(f'family M: n={len(m_group)}, mean={m_group.mean():.2f} MPa, std={m_group.std():.2f} MPa')

result = stats.ttest_ind(c_group, m_group)
print(f't-statistic: {result.statistic:.3f}')
print(f'p-value:     {result.pvalue:.3g}')

family C: n=385, mean=488.27 MPa, std=121.27 MPa
family M: n=191, mean=425.96 MPa, std=82.10 MPa
t-statistic: 6.408
p-value:     3.07e-10


**H0: alloy families C and M have the same mean tensile strength.** p ≈ 3e-10 -- nowhere near
the α = 0.05 line, so we reject H0: this is a real difference, not sampling noise, with a
roughly 62 MPa gap between family means. One *enrichment* step, beyond Midterm 2 scope:
Lecture 28 warned that a p-value alone doesn't say how big a difference is. A standard way
to report size -- new here, shown once, not on the exam -- is Cohen's d, the gap measured
in pooled-standard-deviation units:

In [16]:
pooled_std = np.sqrt(
    ((len(c_group) - 1) * c_group.std() ** 2 + (len(m_group) - 1) * m_group.std() ** 2)
    / (len(c_group) + len(m_group) - 2)
)
d = (c_group.mean() - m_group.mean()) / pooled_std
print(f"Cohen's d: {d:.3f}")

Cohen's d: 0.567


d ≈ 0.57 -- a real, moderate-to-large effect, not just a statistically-detectable sliver.
Purchase-order sentence: "families C and M differ in mean tensile strength by about 62 MPa
(p ≈ 3×10⁻¹⁰), a difference too large and too consistent to be sampling noise." One
sentence tying it back to (c): this is also *why* alloy family alone might be worth adding as
an extra predictor if this model were built out further.

## Checklist, closed

```
[x] look at data       -- (a) describe()
[x] check for outliers -- (b) z-score rule, caught row 626, nothing left after
[x] fit a model        -- (c) six-feature multivariate lstsq fit
[x] evaluate honestly  -- (d) train/test split, five-seed sanity check
[x] test a hypothesis  -- (e) two-sample t-test, family C vs. family M
```

Apart from that one bracketed enrichment aside (Cohen's d), nothing in this section
introduced a new technique -- it chained five techniques you already have. That chaining,
not any single step, is today's actual skill.

# Section 4 -- Quiz 6, in class

Individual, paper or Gradescope (same medium as Quizzes 1-5), scoped **only** to cells you've
actually seen taught -- every item below traces to a specific lecture:

1. Compute the z-score of a flagged point in a short list of numbers and decide whether it
   clears a stated threshold. Traces to **Lecture 27** (the z-score rule) and **Lecture 4**
   (mean/std, computed "by hand").
2. Given a fitted line's slope and intercept, predict `y` at a new `x`. Traces to
   **Lecture 19** (reading slope/intercept in physical units).
3. Given printed train and test R² (or RMSE), diagnose overfitting vs. a healthy gap and name
   the term. Traces to **Lecture 20** (Sections 1-5) and **Lecture 23** (small-sample
   overfitting).
4. Given a p-value and α, state the plain-English conclusion. Traces to **Lecture 28**.

No code required -- same deliberate format as the practice midterm packet from Section 5.
One example of each style, so the *format* isn't a surprise:

In [17]:
vals = np.array([305, 312, 298, 309, 301, 355])
z_vals = (vals - vals.mean()) / vals.std()
print(np.round(z_vals, 2))

[-0.43 -0.07 -0.8  -0.23 -0.64  2.17]


The last value, `z ≈ 2.17`, clears `|z| > 2` but not `|z| > 3` -- exactly the "which
threshold?" judgment Lecture 27 warned is a convention, not a law.

A line `y = 2.0*x + 15.0`: predict `y` at `x = 30` by hand (`2.0 * 30 + 15.0 = 75.0`), then
check it in code:

In [18]:
slope, intercept = 2.0, 15.0
print(slope * 30 + intercept)

75.0


For item 3's style, Lecture 23's own numbers are the real example: a 3D-printing strength
model with 7 features but only 37 training rows scored **train R² = 0.669, test R² = 0.094**
-- a large gap, the small-sample overfitting Lecture 20 warned about, not a healthy one like
today's Section 3(d) split. For item 4: Lecture 28's `abs` vs. `pla` comparison scored
**p = 0.04126** at α = 0.05 -- reject H0, a real but borderline difference, worth saying so in
the plain-English answer rather than just "significant."

# Section 5 -- Midterm 2 format review

Midterm 2 is Lecture 31, two class periods away -- Lecture 30 in between is a dedicated
review day for a full practice packet. Midterm 2 itself: single-day, **on paper**,
closed-notebook, no laptops or AI -- a deliberate course policy, same spirit as Quiz 6 and
Quiz 3's format review before it.

**In scope:** everything from Lecture 19 through Lecture 28 -- the whole checklist from
Section 1: linear/nonlinear/multivariate regression, evaluation and overfitting,
classification basics (kNN), outliers, and hypothesis testing.

**Out of scope:** raw Module 1 syntax not reused here, and anything beyond what Lecture 26
covered (no logistic regression, no decision trees, no neural nets) -- classification and
evaluation are fair game precisely because they were taught as extensions of curve fitting,
never as a separate standalone unit.

You've never taken a programming exam on paper before. Three question types to expect --
same three from Lecture 16/17, now applied to Module 2 material:

1. **Read code, predict output.** A short snippet using `numpy`/`scipy`/Pandas -- write down
   exactly what gets printed.
2. **Write short code by hand.** Full credit for correct logic even with a syntax slip.
3. **Interpret printed output or a printed plot.** A fitted line, a train/test table, or a
   scatter/box plot is printed on the page, no code -- read off a value, judge a fit, or
   diagnose overfitting from the shape of an error curve.

Two concrete rehearsal items, worked here so the format itself isn't new on exam day:

**Rehearsal item A (read code, predict output).** What does this print?

```python
import numpy as np
from scipy import stats

proof_stress = np.array([200, 350, 500])
result = stats.linregress(proof_stress, [340, 480, 610])
print(round(result.slope, 2))
```

In [19]:
import numpy as np
from scipy import stats

proof_stress = np.array([200, 350, 500])
result = stats.linregress(proof_stress, [340, 480, 610])
print(round(result.slope, 2))

0.9


`0.9` -- three points fall almost exactly on a line of slope 0.9 (check by hand: from
`(200, 340)` to `(500, 610)`, `Δy/Δx = 270/300 = 0.9`). The middle point `(350, 480)` sits
a hair off that exact line, which is exactly why `linregress` and not a ruler is doing the
work.

**Rehearsal item B (interpret printed output, no code).** A model fit on 900 alloys reports
**train RMSE = 42 MPa** and **test RMSE = 245 MPa**. In one sentence: what's going wrong, and
what Module 2 term names it?

*Model answer:* the model fits training data far better than it generalizes -- a large,
unhealthy train/test gap -- which is **overfitting**, not a healthy small gap like Section
3(d)'s example above.

You'll leave today with the full practice Midterm 2 packet (Lecture 30's handout) -- same
format and length as the real thing, drawn broadly across Module 2. HW9 is due before
Midterm 2 and is the best remaining practice, since it exercises this exact five-step
pipeline on a different dataset. Office hours before Midterm 2 will be posted on the LMS.

# Wrap-up

* The five ideas of Module 2 -- fit, evaluate, extend to nonlinear/multivariate, catch
  outliers, test a hypothesis -- are one pipeline, not five separate topics.
* "Nothing flagged" and "modest R²" are both legitimate findings, not failed analyses --
  Section 3 hit both today, honestly.
* A coefficient's sign can surprise you without being wrong -- Lecture 23's "holding
  everything else fixed" reading rule is what makes a surprising sign interpretable instead
  of alarming.

**Take-home message:** materials data science isn't five separate tricks -- it's one
repeatable pipeline (look, check, fit, evaluate, test) you run start to finish on whatever
dataset lands on your desk, messy or clean.

Next class: Lecture 30, a dedicated practice day for the full Midterm 2 packet, worked under
light time pressure and debriefed together. Midterm 2 itself is Lecture 31.